In [5]:
import os, sys
os.environ['CUDA_VISIBLE_DEVICES'] = '3,'
from pathlib import Path
import json
from datetime import datetime
from tqdm import tqdm
from omegaconf import OmegaConf, DictConfig, ListConfig
import hydra
import numpy as np
from PIL import Image
import lovely_tensors as lt
import imageio as iio
lt.monkey_patch()
import torch

from hmr4d.dataset.pure_motion.amass import AmassDataset
from hmr4d.dataset.imgfeat_motion.uni3c_aligned import Uni3CAlignedDatasetV1
from hmr4d.dataset.pure_motion.cam_traj_utils_v2 import create_rotation_track, create_translation_track
from hmr4d.utils.body_model import BodyModelSMPLX
from hmr4d.utils.body_model.utils import load_hand_faces_and_colors
from hmr4d.utils.geo_transform import transform_mat
from hmr4d.utils.geo.transforms import axis_rotate_to_matrix
from hmr4d.utils.vis.renderer import Renderer, Renderer_Point
from hmr4d.utils.video_io_utils import get_writer
from hmr4d.utils.image_gen.stable_diffusion import ImageGenerator
import third_party.depth_pro.depth_pro as depth_pro
from third_party.depth_pro.utils import get_boundaries_mask, get_depth_image, get_points_3d
from hmr4d.utils.preproc import Extractor
from hmr4d.utils.preproc.vitfeat_extractor import get_batch

device = 'cuda:0'

In [7]:
dataset = Uni3CAlignedDatasetV1()

[02/12 23:19:11][INFO] [UNI3C_SynthGenerated] Loading from inputs/uni3c_aligned ...
[02/12 23:19:11][INFO] [UNI3C_SynthGenerated] Loaded from 342 synthetic samples
[02/12 23:19:11][INFO] [UNI3C_SynthGenerated] Using 342 synthetic samples


In [ ]:
width, height = 768, 480

dataset = AmassDataset(cam_augmentation='v20', width=width, height=height, f_fullframe=24.0, 
                       motion_frames=120)

smplx = BodyModelSMPLX(
    model_path="inputs/checkpoints/body_models", model_type="smplx",
    gender="neutral", num_pca_comps=12, flat_hand_mean=False,
).to(device)
smplx_colors = torch.tensor(torch.load("hmr4d/utils/body_model/smplx_color.pt")/ 255.)[None]

hand_idxs, hand_faces, hand_colors = load_hand_faces_and_colors(smplx.faces)
hand_faces = hand_faces.to(device)

depth_model, depth_transform = depth_pro.create_model_and_transforms(device=device)
depth_model = depth_model.eval()

extractor = Extractor()

In [ ]:
for _ in range(50):
    output_path = Path("inputs/uni3c_aligned/test_" + datetime.strftime(datetime.now(), "%m%d") + "/test_" + datetime.strftime(datetime.now(), "%m%d%H%M%S"))
    output_path.mkdir(parents=True, exist_ok=True)
    
    index = np.random.randint(len(dataset))

    batch = dataset[index]
    nframe = batch["length"]
    K_fullimg = batch['K_fullimg'][0]
    smpl_params_c = batch['smpl_params_c']
    batch['index'] = index
    torch.save(batch, output_path / "batch_meta.pt")

    verts = smplx(**{k:v.to(device) for k,v in smpl_params_c.items()}).vertices
    joints = smplx(**{k:v.to(device) for k,v in smpl_params_c.items()}).joints
    renderer_c = Renderer(width, height, device="cuda", faces=smplx.faces, K=K_fullimg)

    writer1 = get_writer(output_path / 'smpl_render.mp4', fps=30, crf=23)
    writer2 = get_writer(output_path / 'hand_render.mp4', fps=30, crf=23)
    for j in tqdm(range(nframe), desc=f"Rendering Global"):
        black_backg = np.zeros((height, width, 3)).astype(np.uint8)
        smpl_rgbs, smpl_depths = renderer_c.render_mesh(
            verts[j], background=black_backg, 
            colors=smplx_colors, return_depth=True
        )
        hand_rgbs, hand_depths = renderer_c.render_mesh(
            verts[j, hand_idxs], background=black_backg, faces=hand_faces,
            colors=hand_colors,  return_depth=True
        )
        unify_mask = (hand_depths <= smpl_depths+0.01)[..., None]

        refined_hand_rgbs = np.clip(
            hand_rgbs * unify_mask + np.zeros_like(hand_rgbs) * (1 - unify_mask), 
        0, 255).astype(np.uint8)

        writer1.write_frame(smpl_rgbs)
        writer2.write_frame(refined_hand_rgbs)
    writer1.close()
    writer2.close()

    pred_c_joints, pred_c_valids = renderer_c.project_points_to_full_image(joints[0])
    canvas = renderer_c.render_openpose(pred_c_joints, pred_c_valids)
    Image.fromarray(canvas).save(output_path / "openpose_render.png")

In [ ]:
main_output_dir = Path("inputs/uni3c_aligned/test_0208")
output_dirs = sorted([p for p in main_output_dir.iterdir() if p.is_dir()])
for output_dir in output_dirs:
    batch = torch.load(output_dir / "batch_meta.pt")
    nframe = batch["length"]
    width, height, K_fullimg = batch['meta']['width'], batch['meta']['height'], batch['K_fullimg'][0]
    verts = smplx(**{k:v.to(device) for k,v in batch['smpl_params_c'].items()}).vertices
    joints = smplx(**{k:v.to(device) for k,v in batch['smpl_params_c'].items()}).joints

    final_img = Image.open(output_dir / "reference.png").convert("RGB").resize((width, height))
    prediction = depth_model.infer(depth_transform(np.array(final_img)), f_px=K_fullimg[0,0].to(device))
    depth = torch.clip(prediction["depth"], 1e-4, 100)  # Depth in [m].
    boundary_mask = get_boundaries_mask((1 / (depth + 1e-7))[None, None], sobel_threshold=0.35)[0, 0].reshape(-1).cpu()
    color_depth = get_depth_image(depth.detach().cpu().numpy(), min_depth=0.1, max_depth=250.0)
    color_depth.save(output_dir / "depth.png")
    
    renderer_c = Renderer(width, height, device="cuda", faces=smplx.faces, K=K_fullimg)
    pred_c_joints, pred_c_valids = renderer_c.project_points_to_full_image(joints[0])
    key_3d_index = (pred_c_joints[0,:,1] * width + pred_c_joints[0,:, 0]).numpy().astype(int)
    key_3d_index = key_3d_index[pred_c_valids[0].numpy().astype(bool)]
    depth_avg = depth.reshape(-1)[key_3d_index].median().item()

    points3d = get_points_3d(depth, K_fullimg)
    c2w_0 = torch.eye(4)
    c2w_0[2, 3] = -depth_avg
    R_elevation = axis_rotate_to_matrix(-5.0, 'x', with_trans=True, use_deg=True)[0]
    c2w_0 = R_elevation @ c2w_0
    points3d = (c2w_0.numpy()[:3] @ points3d.T).T
    colors = np.array(final_img).reshape(height * width, 3) / 255.

    c2ws = c2w_0.clone()
    R_tmp = c2ws[:3,:3]
    R_tmp[:, 0] = -R_tmp[:, 0]
    R_tmp[:, 1] = -R_tmp[:, 1]
    T_tmp = c2ws.inverse()[:3, -1]

    R_w2c_tmp, _1 = create_rotation_track(R_tmp, nframe, linspace=batch['meta']['R_linspace'], 
                                          **{k : batch['meta']['cam_meta'][k] for k in ['yaw', 'pitch', 'roll']})
    t_w2c_tmp, _2 = create_translation_track(R_tmp, T_tmp, nframe, linspace=batch['meta']['t_linspace'], 
                                             **{k : batch['meta']['cam_meta'][k] for k in ['tx', 'ty', 'tz']})
    T_w2c_tmp = transform_mat(R_w2c_tmp, t_w2c_tmp)

    cam_info = {
        "intrinsic": K_fullimg.numpy().tolist(), 
        "extrinsic": T_w2c_tmp.numpy().tolist(), 
        "height": height, "width": width
    }
    Path(output_dir / "cam_info.json").write_text(json.dumps(cam_info))

    renderer_pc = Renderer_Point(width, height, K_fullimg, device=device)
    point_cloud = renderer_pc.create_point_cloud(points3d, colors, boundary_mask=boundary_mask)

    writer1 = get_writer(output_dir / 'render.mp4', fps=30, crf=23)
    writer2 = get_writer(output_dir / 'render_mask.mp4', fps=30, crf=23)
    writer3 = get_writer(output_dir / 'combined_render.mp4', fps=30, crf=23)
    for j in tqdm(range(nframe), desc=f"Rendering Global"):
        camera = renderer_pc.create_camera(T_w2c_tmp[j:j+1].to(device))
        render_rgb, render_mask = renderer_pc(point_cloud, camera)
        
        render_smpl_rgbs, render_smpl_masks = renderer_c.render_mesh(
            verts[j], background=np.zeros((height, width, 3)).astype(np.uint8), 
            colors=smplx_colors,
            return_mask=True
        )
        render_fused_smpl_rgbs = np.clip(
            render_smpl_rgbs * render_smpl_masks + render_rgb * (1 - render_smpl_masks), 
        0, 255).astype(np.uint8)

        writer1.write_frame(render_rgb)
        writer2.write_frame(render_mask)
        writer3.write_frame(render_fused_smpl_rgbs)

    writer1.close()
    writer2.close()
    writer3.close()

In [ ]:
# for idx in range(len(ds)):
#     mid = ds.idx2meta[idx]
#     batch = torch.load(ds.root / f"{mid}/batch_meta.pt")
#     with torch.no_grad():
#         gt_verts437, gt_j3d = smplx_coco(**batch["smpl_params_c"])
        
#     i_x2d = safely_render_x3d_K(gt_verts437[None], batch["K_fullimg"][None], thr=0.3)
#     bbx_xys = get_bbx_xys(i_x2d, do_augment=True)[0]
#     batch["bbx_xys"] = bbx_xys
#     torch.save(batch, ds.root / f"{mid}/batch_meta.pt")

In [ ]:
main_video_dir = Path("inputs/uni3c_aligned/")
for video_dir in tqdm(list(main_video_dir.glob("*/*"))[230:]):
    video = iio.mimread(str(video_dir / "final.mp4"))
    output_dir = video_dir / "frames"
    output_dir.mkdir(exist_ok=True, parents=True)
    for frame_id, frame in enumerate(video[1:]):
        Image.fromarray(frame).save(output_dir / f"{frame_id:04d}.png")
    print(video_dir, len(video))

In [ ]:
main_output_dir = Path("inputs/uni3c_aligned/test_0208")
for output_dir in tqdm(list(main_output_dir.glob("*"))):
    batch = torch.load(output_dir / "batch_meta.pt")

    video = iio.mimread(str(output_dir / "final.mp4"))
    video = np.concatenate(np.expand_dims(video[1:], axis=0))
    video, bbx_xys = get_batch(video, batch['bbx_xys'], img_ds=1.0, path_type='np')
    vit_features = extractor.extract_video_features(video, bbx_xys)
    batch['f_imgseq'] = vit_features
    torch.save(batch, output_dir / "batch_meta.pt")

In [ ]:
output_path = Path("outputs/uni3c_aligned/test_" + datetime.strftime(datetime.now(), "%m%d") + "/test_" + datetime.strftime(datetime.now(), "%m%d%H%M%S"))
output_path.mkdir(parents=True, exist_ok=True)

In [ ]:
index = np.random.randint(len(dataset))

batch = dataset[index]
nframe = batch["length"]
K_fullimg = batch['K_fullimg'][0]
smpl_params_c = batch['smpl_params_c']
batch['index'] = index
torch.save(batch, output_path / "batch_meta.pt")

verts = smplx(**{k:v.to(device) for k,v in smpl_params_c.items()}).vertices
joints = smplx(**{k:v.to(device) for k,v in smpl_params_c.items()}).joints

renderer_c = Renderer(width, height, device="cuda", faces=smplx.faces, K=K_fullimg)

pred_c_joints, pred_c_valids = renderer_c.project_points_to_full_image(joints[0])
canvas_1 = renderer_c.render_openpose(pred_c_joints, pred_c_valids)
smpl_rgbs_1 = renderer_c.render_mesh(verts[0], colors=smplx_colors)
pred_c_joints, pred_c_valids = renderer_c.project_points_to_full_image(joints[-1])
canvas_2 = renderer_c.render_openpose(pred_c_joints, pred_c_valids)
smpl_rgbs_2 = renderer_c.render_mesh(verts[-1], colors=smplx_colors)
total_img = Image.fromarray(np.concatenate([
    np.concatenate([smpl_rgbs_1, canvas_1], axis=1),
    np.concatenate([smpl_rgbs_2, canvas_2], axis=1),
], axis=0))

total_img

In [ ]:
renderer_c = Renderer(width, height, device="cuda", faces=smplx.faces, K=K_fullimg)

writer1 = get_writer(output_path / 'smpl_render.mp4', fps=30, crf=23)
writer2 = get_writer(output_path / 'hand_render.mp4', fps=30, crf=23)
for j in tqdm(range(nframe), desc=f"Rendering Global"):
    black_backg = np.zeros((height, width, 3)).astype(np.uint8)
    smpl_rgbs, smpl_depths = renderer_c.render_mesh(
        verts[j], background=black_backg, 
        colors=smplx_colors, return_depth=True
    )
    hand_rgbs, hand_depths = renderer_c.render_mesh(
        verts[j, hand_idxs], background=black_backg, faces=hand_faces,
        colors=hand_colors,  return_depth=True
    )
    unify_mask = (hand_depths <= smpl_depths+0.01)[..., None]

    refined_hand_rgbs = np.clip(
        hand_rgbs * unify_mask + np.zeros_like(hand_rgbs) * (1 - unify_mask), 
    0, 255).astype(np.uint8)

    writer1.write_frame(smpl_rgbs)
    writer2.write_frame(refined_hand_rgbs)
writer1.close()
writer2.close()

pred_c_joints, pred_c_valids = renderer_c.project_points_to_full_image(joints[0])
canvas = renderer_c.render_openpose(pred_c_joints, pred_c_valids)

In [ ]:
prompt = np.random.choice(id_prompts) + np.random.choice(scene_prompts) 
seed = np.random.randint(2**31)
coarse_img, coarse_meta = img_gen.step1(
    prompt=prompt,
    image=Image.fromarray(canvas),
    width=width, height=height, 
    generator = torch.Generator(device='cuda').manual_seed(seed)
)

final_img, final_meta = img_gen.step2(
    prompt=prompt,
    image=coarse_img[0],
    width=width*2, height=height*2,
    generator = torch.Generator(device='cuda').manual_seed(seed)

)
coarse_img[0].save(output_path / "reference_coarse.png")
final_img = final_img[0].resize((width, height))
final_img.save(output_path / "reference.png")
(output_path / "img_meta.json").write_text(json.dumps({'coarse_meta': coarse_meta, 'final_meta': final_meta}, indent=4))
final_img

In [ ]:
prediction = depth_model.infer(depth_transform(np.array(final_img)), f_px=K_fullimg[0,0].to(device))
depth = torch.clip(prediction["depth"], 1e-4, 100)  # Depth in [m].

key_3d_index = (pred_c_joints[0,:,1] * width + pred_c_joints[0,:, 0]).numpy().astype(int)
key_3d_index = key_3d_index[pred_c_valids[0].numpy().astype(bool)]
depth_avg = depth.reshape(-1)[key_3d_index].median().item()

boundary_mask = get_boundaries_mask((1 / (depth + 1e-7))[None, None], sobel_threshold=0.35)[0, 0].reshape(-1).cpu()

color_depth = get_depth_image(depth.detach().cpu().numpy(), min_depth=0.1, max_depth=250.0)
color_depth.save(output_path / "depth.png")
color_depth

In [ ]:
start_elevation = 5.0

points3d = get_points_3d(depth, K_fullimg)
c2w_0 = torch.eye(4)
c2w_0[2, 3] = -depth_avg
R_elevation = axis_rotate_to_matrix(-start_elevation, 'x', with_trans=True, use_deg=True)[0]
c2w_0 = R_elevation @ c2w_0
points3d = (c2w_0.numpy()[:3] @ points3d.T).T
colors = np.array(final_img).reshape(height * width, 3) / 255.

c2ws = c2w_0.clone()
R_tmp = c2ws[:3,:3]
R_tmp[:, 0] = -R_tmp[:, 0]
R_tmp[:, 1] = -R_tmp[:, 1]
T_tmp = c2ws.inverse()[:3, -1]

R_w2c_tmp, _1 = create_rotation_track(R_tmp, nframe, linspace=batch['meta']['R_linspace'], **{k : batch['meta']['cam_meta'][k] for k in ['yaw', 'pitch', 'roll']})
t_w2c_tmp, _2 = create_translation_track(R_tmp, T_tmp, nframe, linspace=batch['meta']['t_linspace'], **{k : batch['meta']['cam_meta'][k] for k in ['tx', 'ty', 'tz']})
T_w2c_tmp = transform_mat(R_w2c_tmp, t_w2c_tmp)

cam_info = {
    "intrinsic": K_fullimg.numpy().tolist(), 
    "extrinsic": T_w2c_tmp.numpy().tolist(), 
    "height": height, "width": width
}
Path(output_path / "cam_info.json").write_text(json.dumps(cam_info))

In [ ]:
renderer_pc = Renderer_Point(width, height, K_fullimg, device=device)
point_cloud = renderer_pc.create_point_cloud(points3d, colors, boundary_mask=boundary_mask)

writer1 = get_writer(output_path / 'render.mp4', fps=30, crf=23)
writer2 = get_writer(output_path / 'render_mask.mp4', fps=30, crf=23)
writer3 = get_writer(output_path / 'combined_render.mp4', fps=30, crf=23)

for j in tqdm(range(nframe), desc=f"Rendering Global"):
    camera = renderer_pc.create_camera(T_w2c_tmp[j:j+1].to(device))
    render_rgb, render_mask = renderer_pc(point_cloud, camera)
    
    render_smpl_rgbs, render_smpl_masks = renderer_c.render_mesh(
        verts[j], background=np.zeros((height, width, 3)).astype(np.uint8), 
        colors=smplx_colors,
        return_mask=True
    )
    render_fused_smpl_rgbs = np.clip(
        render_smpl_rgbs * render_smpl_masks + render_rgb * (1 - render_smpl_masks), 
    0, 255).astype(np.uint8)

    writer1.write_frame(render_rgb)
    writer2.write_frame(render_mask)
    writer3.write_frame(render_fused_smpl_rgbs)

writer1.close()
writer2.close()
writer3.close()

In [ ]:
# motion_files = torch.load("inputs/AMASS/hmr4d_support/smplxpose_v2.pth")
# seqs = {k: v for k,v in motion_files.items() if 'moyo_smplxn' not in k and v['pose'].shape[0] >= 25}
# print(f"Total motion files: {len(seqs):,}")

# ### AMASS Train Dataset --Load Data-- ###
# key = 'inputs/smplx_amass/smplxn_raw/Transitions/Transitions/mazen_c3d/sit_jumpinplace_stageii.npz'
# start, end = 92, 212
# raw_data = seqs[key]
# raw_len = raw_data["pose"].shape[0]
# nframe = end - start

# data = {
#     "body_pose": raw_data["pose"][start:end, 3:],  # (F, 63)
#     "betas": raw_data["beta"].repeat(end-start, 1),  # (10)
#     "global_orient": raw_data["pose"][start:end, :3],  # (F, 3)
#     "transl": raw_data["trans"][start:end, :3],  # (F, 3)
#     "data_name" : "amass"
# }
# # data = interpolate_smpl_params(data, motion_frames_len)
# data["global_orient"], data["transl"], _ = get_tgtcoord_rootparam(
#     data["global_orient"], data["transl"], tsf="az->ay",
# )
# global_orient_w, transl_w = rotate_around_axis(data["global_orient"], data["transl"], axis="y")
# smpl_params_w = {
#     'body_pose' : data["body_pose"], 'betas': data["betas"], 
#     'global_orient': global_orient_w, 'transl': transl_w
# }
# w_j3d = smplx(**{k:v[0:1].to(device) for k,v in smpl_params_w.items()}).joints.cpu()

# # kp3d = smplx_coco(**{k: v.to(device) for k,v in smpl_params_w.items()})
# # np.save("coco_17joints.npy", kp3d.cpu().float().numpy())
# # smplx_out = smplx(**{k: v.to(device) for k,v in smpl_params_w.items()})
# # np.save("full_vertices.npy", smplx_out.vertices.cpu().float().numpy())
# # np.save("full_joints.npy", smplx_out.joints.cpu().float().numpy())

# width, height, f_fullframe = 768, 480, 24
# width, height, K_fullimg = create_camera_sensor(width, height, f_fullframe)

# np.random.seed(256)
# R0_w2c, t0_w2c = create_camera(w_j3d[0,0], width, K_fullimg[0,0])  # (3, 3) and (3,)
# print(t0_w2c.v)

# R0_new_w2c, t0_new_w2c = adjust_camera_for_rel_area(
#     w_j3d, R0_w2c, t0_w2c, K_fullimg, 
#     0.005, 0.15, l_margin=0.4, t_margin=0.6, r_margin=0.8, b_margin=0.8)
# print(t0_new_w2c.v)

# T0_w2c = transform_mat(R0_new_w2c, t0_new_w2c)

# R_w2c, R_linspace = create_rotation_track(R0_new_w2c, nframe, yaw=5)
# t_w2c, t_linspace = create_translation_track(R0_new_w2c, t0_new_w2c, nframe, tx=0.5)
# T_w2c = transform_mat(R_w2c, t_w2c)
# cam_angvel = compute_cam_angvel(T_w2c[:, :3, :3])  # (F, 6)

# offset = smplx.get_skeleton(data["betas"][0].to(device))[0]  # (3)
# global_orient_c, transl_c = get_c_rootparam(global_orient_w, transl_w, T_w2c, offset.cpu())
# smpl_params_c = {
#     "body_pose": data["body_pose"],  # (F, 63)
#     "betas": data["betas"],  # (F, 10)
#     "global_orient": global_orient_c,  # (F, 3)
#     "transl": transl_c,  # (F, 3)
# }
# verts = smplx(**{k:v.to(device) for k,v in smpl_params_c.items()}).vertices
# joints = smplx(**{k:v.to(device) for k,v in smpl_params_c.items()}).joints

##################
# data = dataset._load_data(index)
# nframe = data['length']
# body_pose = data["body_pose"]
# betas = data["betas"]
# global_orient_w, transl_w = rotate_around_axis(data["global_orient"], data["transl"], axis="y")
# smpl_params_w = {
#     "body_pose": body_pose,  # (F, 63)
#     "betas": betas,  # (F, 10)
#     "global_orient": global_orient_w,  # (F, 3)
#     "transl": transl_w,  # (F, 3)
# }
# w_j3d = smplx(**{k:v.to(device) for k,v in smpl_params_w.items()}).joints.cpu()

# cam_augmentor = CameraAugmenterV20(768, 480, f_fullframe=24)
# T_w2c, R_linspace, t_linspace, meta = cam_augmentor(w_j3d, seed=seed)
# K_fullimg = cam_augmentor.K_fullimg



In [ ]:

# data_cfg = OmegaConf.load("hmr4d/configs/data/mocap/trainX_testY.yaml")
# data_cfg.dataset_opts.train = DictConfig(
#     {
#         'pure_motion_amass': DictConfig({
#         '_target_': 'hmr4d.dataset.pure_motion.amass.AmassDataset', 
#         'cam_augmentation': 'v20'})
#     }
# )
# data_cfg.dataset_opts.val = DictConfig(
#     {
#         'emdb': DictConfig({
#         '_target_': 'hmr4d.dataset.emdb.emdb_motion_test.EmdbSmplFullSeqDataset', 
#         'flip_test': True})
#     }
# )

# datamodule = hydra.utils.instantiate(data_cfg, _recursive_=False)
# dataloader = datamodule.train_dataloader()

# for batch in tqdm(dataloader):
#     vid_list = [x['vid'] for x in batch['meta']]
#     if 'ACCAD/ACCAD/Male2Running_c3d/C23_-_put_down_box_to_run_stageii.npz' in vid_list:
    
#     for name,x in batch.items():
#         if name not in ['meta', 'B']:
#             if x is None:
#                 raise RuntimeError(f"{name} is None: {x}")
#             if type(x) is dict:
#                 for k,v in x.items():
#                     if not torch.isfinite(v).all():
#                         raise RuntimeError(f"{name}.{k} is not finite: {v}")
#             elif not torch.isfinite(x).all():
#                 raise RuntimeError(f"{name} is not finite: {x}")
